In [1]:
# load in pandas df in ../..//Week10/PurpleData.pkl 
import pandas as pd

df = pd.read_pickle('../../Week10/PurpleData.pkl')
df.head()

,R,G,B,Class
0,221.0,62.0,205.0,1
1,157.0,47.0,181.0,1
2,128.0,0.0,255.0,1
3,148.0,0.0,211.0,1
4,145.0,55.0,213.0,1


In [12]:
df['Class'].value_counts()

1    57
0    49
Name: Class, dtype: int64

# Build a fully connected Pytorch NN to Classify Class from R, G and B in df

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # Define the neural network
# class SimpleNN(nn.Module):
#     def __init__(self, input_size=3, hidden_size=3, dropout_prob=0.5):
#         super(SimpleNN, self).__init__()
#         # Input to hidden layer
#         self.fc1 = nn.Linear(input_size, hidden_size)
#         # Dropout layer
#         self.dropout = nn.Dropout(p=dropout_prob)
#         # Hidden to output layer (single output for binary classification)
#         self.fc2 = nn.Linear(hidden_size, 1)
    
#     def forward(self, x):
#         # Input to hidden layer with ReLU activation
#         x = F.relu(self.fc1(x))
#         # Apply dropout
#         x = self.dropout(x)
#         # Hidden to output layer with sigmoid activation
#         # x = torch.sigmoid(self.fc2(x))
#         x = self.fc2(x)
#         return x

# # Instantiate the model
# input_size = 3  # Number of input features (R, G, B)
# hidden_size = 3  # Number of neurons in the hidden layer
# dropout_prob = 0.5  # Dropout probability

# model = SimpleNN(input_size, hidden_size, dropout_prob)

# # Print the model architecture
# print(model)

SimpleNN(
  (fc1): Linear(in_features=3, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=3, out_features=1, bias=True)
)


In [11]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F

# Load the dataset
df = pd.read_pickle('../../Week10/PurpleData.pkl')

# Define the neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size=3, hidden_size=3, dropout_prob=0.5):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout = nn.Dropout(p=dropout_prob)
        self.fc2 = nn.Linear(hidden_size, 1)  # Single output for binary classification
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # No sigmoid here; handled by BCEWithLogitsLoss
        return x

# Preprocess the data
# Normalize the input features to the range [0, 1]
X = df[['R', 'G', 'B']].values / 255.0  # Assuming RGB values are in the range [0, 255]
y = df['Class'].values          # Target column (binary: 0/1)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Add dimension for binary classification

# Create a DataLoader for batching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)  # Batch size of 16

# Instantiate the model
input_size = 3  # Number of input features (R, G, B)
hidden_size = 3  # Number of neurons in the hidden layer
dropout_prob = 0.5  # Dropout probability
model = SimpleNN(input_size, hidden_size, dropout_prob)

print(model)

# Define the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Combines sigmoid and binary cross-entropy
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20  # Number of epochs
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    for batch_X, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss for each epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Save the trained model
torch.save(model.state_dict(), 'simple_nn_model.pth')
print("Model training complete and saved.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    outputs = model(X_tensor)
    predictions = torch.sigmoid(outputs).round()  # Apply sigmoid and round to get binary predictions
    accuracy = (predictions.eq(y_tensor).sum().item()) / y_tensor.size(0)
    print(f"Model Accuracy: {accuracy:.4f}")

SimpleNN(
  (fc1): Linear(in_features=3, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=3, out_features=1, bias=True)
)
Epoch [1/20], Loss: nan
Epoch [2/20], Loss: nan
Epoch [3/20], Loss: nan
Epoch [4/20], Loss: nan
Epoch [5/20], Loss: nan
Epoch [6/20], Loss: nan
Epoch [7/20], Loss: nan
Epoch [8/20], Loss: nan
Epoch [9/20], Loss: nan
Epoch [10/20], Loss: nan
Epoch [11/20], Loss: nan
Epoch [12/20], Loss: nan
Epoch [13/20], Loss: nan
Epoch [14/20], Loss: nan
Epoch [15/20], Loss: nan
Epoch [16/20], Loss: nan
Epoch [17/20], Loss: nan
Epoch [18/20], Loss: nan
Epoch [19/20], Loss: nan
Epoch [20/20], Loss: nan
Model training complete and saved.
Model Accuracy: 0.0000
